In [ ]:
import pandas as pd
import numpy as np

## Read data from W&Bs
The CSV being read into pandas is from a W&Bs' report.

In [ ]:
dataframe = pd.read_csv('results-data/wandb_export_2025-07-24T20_33_32.270-04_00.csv')

In [ ]:
N = 10

methods = [
    'eval-supervised-no-gnn', 'eval-supervised-v2-encoder',
    'eval-linear-probe-v2-encoder', 'eval-fine-tune-v2-encoder',
    'eval-linear-probe-v2-encoder-barlow-twins', 'eval-fine-tune-v2-encoder-barlow-twins'
    ]

precision_matrix = None
recall_matrix = None
f1_matrix = None

for method in methods:
    method_df = dataframe[dataframe.Name.str.contains(method)]

    if method in ('eval-linear-probe-v2-encoder', 'eval-fine-tune-v2-encoder'):
        method_df = method_df[~method_df.Name.str.contains('barlow-twins')]
    
    assert method_df.shape[0] == N

    prec_values = method_df.eval_precision.values[np.newaxis, :]
    recall_values = method_df.eval_recall.values[np.newaxis, :]
    f1_values = method_df.eval_f_score.values[np.newaxis, :]

    precision_matrix = prec_values if precision_matrix is None \
        else np.concatenate([precision_matrix, prec_values], axis=0)
    recall_matrix = recall_values if recall_matrix is None \
        else np.concatenate([recall_matrix, recall_values], axis=0)
    f1_matrix = f1_values if f1_matrix is None \
        else np.concatenate([f1_matrix, f1_values], axis=0)

In [ ]:
print(f"Precision Means: {precision_matrix.mean(axis=1)}")
print(f"Recall Means: {recall_matrix.mean(axis=1)}")
print(f"F1 Means: {f1_matrix.mean(axis=1)}")

In [ ]:
# from scipy.stats import f_oneway
from scipy.stats import ttest_ind

# Compare No GNN (index=0) vs. GNN Supervised (index=1) using T-Test
# Alternate Hypothesis: Mean performance of supervised model with GNN is greater than mean performance without GNN.
no_gnn_vs_gnn_test = ttest_ind(f1_matrix[1, :], f1_matrix[0, :], alternative='greater')

# Compare GNN Supervised (index=2) vs. GNN Linear Probe (index=1)
# Alternate Hypothesis: Mean performance of supervised GNN model is less than mean performance of linear probe model

# Alternate Hypothesis New: Mean performance of supervised GNN model is greater than mean performance of linear probe model
gnn_vs_linear_probe_ttest = ttest_ind(f1_matrix[1, :], f1_matrix[2, :], alternative='greater')

# Compare GNN Supervised (index=2) vs. GNN Fine-Tune (index=0)
# Alternate Hypothesis: Mean performance of fine-tuned GNN model is greater than mean performance of supervised GNN
gnn_vs_fine_tune_ttest = ttest_ind(f1_matrix[3, :], f1_matrix[1, :], alternative='greater')

print(f"No GNN vs. GNN supervised model: {no_gnn_vs_gnn_test}")
print(f"GNN supervised vs. linear probe model: {gnn_vs_linear_probe_ttest}")
print(f"GNN supervised vs. fine-tuned model: {gnn_vs_fine_tune_ttest}")

# prec_stats = f_oneway(precision_matrix[0, :], precision_matrix[1, :],
#                       precision_matrix[2, :], precision_matrix[3, :])
# recall_stats = f_oneway(recall_matrix[0, :], recall_matrix[1, :],
#                         recall_matrix[2, :], recall_matrix[3, :])
# f1_stats = f_oneway(f1_matrix[0, :], f1_matrix[1, :],
#                     f1_matrix[2, :], f1_matrix[3, :])

# print(f"Precision stats: {prec_stats}")
# print(f"Recall stats: {recall_stats}")
# print(f"F1 score stats: {f1_stats}")